# AYUDA AUTOMÁTICA ANTE ERRORES CON STACK OVERFLOW

Aitor Madoz González. Proyecto del Máster en Big Data Science e Inteligencia Artificial (Universidad de Navarra), asignatura de Técnicas de Recogida de Datos.

En este proyecto hemos creado la función get_SOS_help, que actúa como ayudante cuando un código falla. La función recibe un trozo de código en forma de texto y lo ejecuta. Si funciona, muestra el mensaje "works as intended". Si falla, captura el mensaje de error, lo busca en Stack Overflow, muestra el primer enlace que aparece y la respuesta más votada de esa pregunta, y abre el enlace en el navegador.

Lo hemos hecho paso a paso, probando primero cada parte a mano y convirtiéndola después en una función. El error se captura ejecutando el código dentro de un try/except y guardando el tipo de error junto con su mensaje. La búsqueda se hace con Selenium, que abre un Chrome real y entra en el buscador de Stack Overflow como lo haría una persona; cuando aparece el captcha, el notebook se detiene hasta que lo resolvemos a mano. Después entramos en la primera pregunta y, con BeautifulSoup, leemos su HTML, comparamos los votos de todas las respuestas y nos quedamos con la más votada. Al final todas las piezas se juntan en get_SOS_help, que cierra el navegador automatizado, muestra el resultado de forma ordenada y abre el enlace en el navegador del usuario.

Durante el desarrollo surgieron varios problemas. Primero intentamos trabajar en Google Colab, pero allí el código se ejecuta en un servidor y no se puede ver la ventana de Chrome, así que era imposible resolver el captcha; por eso pasamos a trabajar en local con VS Code. Después, al resolver el captcha la verificación no terminaba de pasar, porque Stack Overflow detectaba que Chrome estaba controlado por Selenium; lo solucionamos con dos opciones de Chrome que ocultan la marca de navegador automatizado. La primera búsqueda con el mensaje de error completo no devolvía ningún resultado, porque incluía la errata concreta de nuestro código ('siz'), que nadie más ha escrito; lo resolvimos quitando del mensaje todo lo que va entre comillas antes de buscar. También tuvimos un error al pulsar Enter antes de resolver el captcha, porque no había resultados y el enlace quedaba vacío, así que añadimos una comprobación. Por último, la respuesta salía desordenada, con el código partido palabra a palabra y con el texto de los botones de la web pegado, y lo arreglamos limpiando el HTML antes de extraer el texto.

Con el error de ejemplo (siz en lugar de size) la función devuelve exactamente la pregunta de Stack Overflow que trata ese error, junto con su respuesta más votada. La principal limitación es que quitar lo que va entre comillas a veces deja la búsqueda demasiado genérica: con el error de 'nn' el primer enlace no tiene relación con el problema. Además, el captcha requiere intervención humana y no se comprueba si la respuesta es correcta, solo se devuelve la más votada. Este notebook debe ejecutarse en local (por ejemplo en VS Code o Jupyter), porque necesita abrir una ventana de Chrome.

### INSTALAMOS LAS LIBRERÍAS

In [10]:
%pip install -q selenium beautifulsoup4 requests numpy


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip3.14 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### IMPORTAMOS LAS LIBRERÍAS Y DEFINIMOS LOS PARÁMETROS

In [11]:
import re
import time
import webbrowser
from urllib.parse import quote_plus

import numpy as np
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By

SO_SEARCH_URL = "https://stackoverflow.com/search?q="
WAIT_SECONDS = 3

### CAPTURAMOS EL MENSAJE DE ERROR

In [12]:
def capturar_error(command):
    try:
        exec(command)
        return None
    except Exception as e:
        return f"{type(e).__name__}: {e}"

print(capturar_error('np.random.uniform(-1, 1, siz=100)'))
print(capturar_error('np.random.uniform(-1, 1, size=100)'))

TypeError: uniform() got an unexpected keyword argument 'siz'
None


### PROBAMOS A MANO LA BÚSQUEDA EN STACK OVERFLOW

In [13]:
def quitar_texto_entre_comillas(error_message):
    sin_comillas = re.sub(r"'[^']*'", "", error_message)
    return re.sub(r"\s+", " ", sin_comillas).strip()

error_message = capturar_error('np.random.uniform(-1, 1, siz=100)')
busqueda = quitar_texto_entre_comillas(error_message)
print(busqueda)

options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
driver = webdriver.Chrome(options=options)

driver.get(SO_SEARCH_URL + quote_plus(busqueda))

input("Si aparece el captcha, resuélvelo en Chrome y pulsa Enter aquí...")
time.sleep(WAIT_SECONDS)

resultados = driver.find_elements(By.CSS_SELECTOR, ".s-post-summary--content-title a")
print(len(resultados))
for resultado in resultados[:3]:
    print(resultado.text, "→", resultado.get_attribute("href"))

TypeError: uniform() got an unexpected keyword argument
10
TypeError: uniform() got an unexpected keyword argument 'low'/'size' → https://stackoverflow.com/questions/72537485/typeerror-uniform-got-an-unexpected-keyword-argument-low-size?r=SearchResults
variance_scaling_initializer() got an unexpected keyword argument 'distribution' → https://stackoverflow.com/questions/52032553/variance-scaling-initializer-got-an-unexpected-keyword-argument-distribution?r=SearchResults
TypeError: function() got an unexpected keyword argument 'njobs' → https://stackoverflow.com/questions/60049343/typeerror-function-got-an-unexpected-keyword-argument-njobs?r=SearchResults


### CONVERTIMOS LA BÚSQUEDA EN FUNCIONES

In [14]:
def crear_navegador():
    options = webdriver.ChromeOptions()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    return webdriver.Chrome(options=options)


def buscar_primer_enlace(driver, error_message):
    busqueda = quitar_texto_entre_comillas(error_message)
    driver.get(SO_SEARCH_URL + quote_plus(busqueda))
    input("Si aparece el captcha, resuélvelo en Chrome y pulsa Enter aquí...")
    time.sleep(WAIT_SECONDS)

    resultados = driver.find_elements(By.CSS_SELECTOR, ".s-post-summary--content-title a")
    if not resultados:
        return None
    return resultados[0].get_attribute("href").split("?")[0]


link = buscar_primer_enlace(driver, error_message)
print(link)

https://stackoverflow.com/questions/72537485/typeerror-uniform-got-an-unexpected-keyword-argument-low-size


### VEMOS LAS RESPUESTAS DE LA PREGUNTA Y SUS VOTOS

In [15]:
print(link)

if link is None:
    print("No se han encontrado resultados: comprueba que el captcha se ha resuelto antes de pulsar Enter.")
else:
    driver.get(link)
    time.sleep(WAIT_SECONDS)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    respuestas = soup.find_all("div", class_="answer")

    print(len(respuestas))
    for respuesta in respuestas:
        print(respuesta["data-score"])

https://stackoverflow.com/questions/72537485/typeerror-uniform-got-an-unexpected-keyword-argument-low-size
2
1
0


### SACAMOS LA RESPUESTA MÁS VOTADA

In [16]:
def obtener_respuesta_mas_votada(driver, link):
    driver.get(link)
    time.sleep(WAIT_SECONDS)
    soup = BeautifulSoup(driver.page_source, "html.parser")

    respuestas = soup.find_all("div", class_="answer")
    if not respuestas:
        return "La pregunta todavía no tiene respuestas.", 0

    mejor_respuesta = respuestas[0]
    for respuesta in respuestas:
        if int(respuesta["data-score"]) > int(mejor_respuesta["data-score"]):
            mejor_respuesta = respuesta

    cuerpo = mejor_respuesta.find("div", class_="js-post-body")
    for boton in cuerpo.find_all("button"):
        boton.decompose()
    partes = []
    for bloque in cuerpo.find_all(["p", "li", "pre"]):
        if bloque.name == "pre":
            partes.append(bloque.get_text().strip())
        elif bloque.find_parent("pre") is None and bloque.find("p") is None:
            partes.append(bloque.get_text(" ", strip=True))

    return "\n\n".join(partes), int(mejor_respuesta["data-score"])


texto, votos = obtener_respuesta_mas_votada(driver, link)
print(f"Votos: {votos}\n")
print(texto)

Votos: 1

We must use random.uniform while importing the related class with an alias (using 'as') or else just use import numpy while importing

An example for using alias is :

from numpy import random as np_random

Then utilize np_random.uniform()

(figured it out and had to use this to solve the problem - note I ran it on colab )


### ABRIMOS EL ENLACE EN EL NAVEGADOR

In [17]:
def prompt_web(link):
    webbrowser.open(link)

prompt_web(link)
driver.quit()

### PRESENTAMOS EL RESULTADO DE FORMA ORDENADA

In [18]:
def mostrar_resultado(error_message, link, texto, votos):
    linea = "═" * 80
    print(linea)
    print("ERROR DETECTADO")
    print(f"    {error_message}")
    print(linea)
    print("PRIMER ENLACE EN STACK OVERFLOW")
    print(f"    {link}")
    print(linea)
    print(f"RESPUESTA MÁS VOTADA ({votos} votos)")
    print()
    print(texto)
    print(linea)

### FUNCIÓN FINAL: get_SOS_help

In [19]:
def get_SOS_help(command):
    error_message = capturar_error(command)
    if error_message is None:
        print("works as intended")
        return

    print(f"Error detectado: {error_message}")
    print("Buscando en Stack Overflow... (resuelve el captcha si aparece)\n")

    driver = crear_navegador()
    try:
        link = buscar_primer_enlace(driver, error_message)
        if link is None:
            print("No se han encontrado resultados en Stack Overflow.")
            return
        texto, votos = obtener_respuesta_mas_votada(driver, link)
    finally:
        driver.quit()

    mostrar_resultado(error_message, link, texto, votos)
    prompt_web(link)

### PRUEBA 1: ARGUMENTO MAL ESCRITO (siz)

In [20]:
get_SOS_help('np.random.uniform(-1, 1, siz=100)')

Error detectado: TypeError: uniform() got an unexpected keyword argument 'siz'
Buscando en Stack Overflow... (resuelve el captcha si aparece)

════════════════════════════════════════════════════════════════════════════════
ERROR DETECTADO
    TypeError: uniform() got an unexpected keyword argument 'siz'
════════════════════════════════════════════════════════════════════════════════
PRIMER ENLACE EN STACK OVERFLOW
    https://stackoverflow.com/questions/72537485/typeerror-uniform-got-an-unexpected-keyword-argument-low-size
════════════════════════════════════════════════════════════════════════════════
RESPUESTA MÁS VOTADA (1 votos)

We must use random.uniform while importing the related class with an alias (using 'as') or else just use import numpy while importing

An example for using alias is :

from numpy import random as np_random

Then utilize np_random.uniform()

(figured it out and had to use this to solve the problem - note I ran it on colab )
════════════════════════════════

### PRUEBA 2: LIBRERÍA MAL ESCRITA (nn)

In [21]:
get_SOS_help('nn.random.uniform(-1, 1, size=100)')

Error detectado: NameError: name 'nn' is not defined
Buscando en Stack Overflow... (resuelve el captcha si aparece)

════════════════════════════════════════════════════════════════════════════════
ERROR DETECTADO
    NameError: name 'nn' is not defined
════════════════════════════════════════════════════════════════════════════════
PRIMER ENLACE EN STACK OVERFLOW
    https://stackoverflow.com/questions/954834/how-do-i-use-raw-input-in-python-3
════════════════════════════════════════════════════════════════════════════════
RESPUESTA MÁS VOTADA (1173 votos)

Starting with Python 3, raw_input() was renamed to input() .

From What’s New In Python 3.0, Builtins section second item.
════════════════════════════════════════════════════════════════════════════════


### PRUEBA 3: CÓDIGO QUE FUNCIONA

In [22]:
get_SOS_help('np.random.uniform(-1, 1, size=100)')

works as intended
